# Multi‑Armed Bandit for Funding Projects — From First Principles

_Generated: 2025-10-22T17:49:38.309738Z_

This notebook simulates **funding decisions** over a set of proposed projects using classic **multi‑armed bandit** strategies. Each project (arm) has a true but unknown **success probability** and a **return per success** (ROI). We compare **ε‑greedy**, **UCB1**, and **Thompson Sampling**. You can seed the system with **past successes/failures** as priors.

**Highlights**
- Arms = projects with (true) success rate `pᵢ`, per‑success payoff `Rᵢ`, and optional cost `Cᵢ` per attempt.
- Rewards per funding: `r = 1{success}·Rᵢ − Cᵢ` (bounded below by `−Cᵢ`).
- Strategies: ε‑greedy (on sample mean), UCB1 (on scaled rewards), Thompson Sampling (Beta priors over `pᵢ`).
- **Past data**: set `(successes_i, failures_i)` to encode prior evidence for project `i`.
- Plots: cumulative reward, cumulative regret, running allocation shares, posterior summaries.
- Artifacts with a one‑click download cell.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports & Helper Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 2025
rng = np.random.default_rng(SEED)

def running_mean(x, w=50):
    x = np.asarray(x, dtype=float)
    n = len(x)
    out = np.zeros(n)
    s = 0.0
    for i in range(n):
        s += x[i]
        j0 = max(0, i-w+1)
        out[i] = (s - (0 if j0==0 else np.sum(x[:j0]))) / (i - j0 + 1)
    return out

## 2) Project Portfolio (Arms)

Define the **true environment** for simulation: success probabilities `p_true`, per‑success ROI `R`, and per‑attempt cost `C`. `past_successes` and `past_failures` seed the priors for Thompson Sampling and the initial empirical means.

In [ ]:
# Example portfolio: 5 projects
p_true = np.array([0.12, 0.20, 0.08, 0.30, 0.18])  # true success probabilities (hidden to algorithms)
R = np.array([100.0, 60.0, 200.0, 50.0, 80.0])     # ROI per success (known to planner)
C = np.array([10.0,  8.0,  15.0,  6.0,  9.0])      # per-attempt cost (known)
names = [f"P{i+1}" for i in range(len(p_true))]

# Prior evidence (can be zeros). These act like pseudo‑counts.
past_successes = np.array([2, 10, 1, 5, 3], dtype=float)
past_failures  = np.array([18, 40, 9, 15, 17], dtype=float)

# Horizon and runs
T = 2000  # number of funding rounds
print("Configured", len(p_true), "projects; horizon T =", T)

# Oracle for regret baselines: best expected net reward per trial
mu_true = p_true * R - C
best_arm = int(np.argmax(mu_true))
best_mu = float(np.max(mu_true))
print("Oracle best arm:", names[best_arm], "with E[reward] =", best_mu)

## 3) Reward Model (Success with ROI minus Cost)

In [ ]:
def pull_arm(i, rng):
    # Bernoulli success with prob p_true[i]; reward = success*R[i] - C[i]
    success = 1.0 if rng.random() < p_true[i] else 0.0
    return success * R[i] - C[i], success

def simulate_single(strategy, T, seed=None):
    local_rng = np.random.default_rng(seed if seed is not None else rng.integers(0, 1<<31))
    return strategy.run(T, local_rng)

## 4) Strategies

In [ ]:
class StrategyBase:
    def __init__(self, k, R, C, names):
        self.k = k; self.R = R; self.C = C; self.names = names
    def run(self, T, rng):
        raise NotImplementedError

### 4.1 ε‑Greedy (on empirical mean reward)

In [ ]:
class EpsilonGreedy(StrategyBase):
    def __init__(self, k, R, C, names, epsilon=0.1, init_mean=0.0):
        super().__init__(k, R, C, names)
        self.epsilon = epsilon
        self.init_mean = init_mean

    def run(self, T, rng):
        k = self.k
        counts = np.zeros(k, dtype=int)
        mean_reward = np.full(k, self.init_mean, dtype=float)

        rewards = np.zeros(T)
        choices = np.zeros(T, dtype=int)
        successes = np.zeros(T)

        for t in range(T):
            if rng.random() < self.epsilon:
                a = int(rng.integers(0, k))
            else:
                a = int(np.argmax(mean_reward))
            r, s = pull_arm(a, rng)
            counts[a] += 1
            # incremental mean update
            mean_reward[a] += (r - mean_reward[a]) / counts[a]
            rewards[t] = r; choices[t] = a; successes[t] = s
        return {"rewards": rewards, "choices": choices, "successes": successes,
                "counts": counts, "mean_reward": mean_reward}

### 4.2 UCB1 (on scaled rewards)

UCB1 assumes rewards in [0,1]. We scale rewards linearly using a known bound. Here a safe bound is `r ∈ [−C_max, R_max]`. We map to [0,1] for index computation and update using the unscaled rewards for evaluation.

In [ ]:
class UCB1(StrategyBase):
    def __init__(self, k, R, C, names):
        super().__init__(k, R, C, names)
        self.Rmax = float(np.max(R))
        self.Cmax = float(np.max(C))
        # scale: r_scaled = (r + Cmax) / (Rmax + Cmax) ∈ [0,1]
        self.scale = self.Rmax + self.Cmax

    def run(self, T, rng):
        k = self.k
        counts = np.zeros(k, dtype=int)
        sum_scaled = np.zeros(k, dtype=float)

        rewards = np.zeros(T)
        choices = np.zeros(T, dtype=int)
        successes = np.zeros(T)

        # pull each arm once to initialize
        for a in range(k):
            r, s = pull_arm(a, rng)
            rewards[a] = r; choices[a] = a; successes[a] = s
            counts[a] += 1
            sum_scaled[a] += (r + self.Cmax) / self.scale

        for t in range(k, T):
            total = np.sum(counts)
            means = sum_scaled / np.maximum(counts, 1)
            bonus = np.sqrt(2.0 * np.log(total) / np.maximum(counts, 1))
            idx = means + bonus
            a = int(np.argmax(idx))
            r, s = pull_arm(a, rng)
            counts[a] += 1
            sum_scaled[a] += (r + self.Cmax) / self.scale
            rewards[t] = r; choices[t] = a; successes[t] = s

        return {"rewards": rewards, "choices": choices, "successes": successes,
                "counts": counts}

### 4.3 Thompson Sampling (Beta‑Bernoulli on success rate)

We place a **Beta prior** on success probability `pᵢ`. For each round we sample `\tilde{p}_i ~ Beta(α_i, β_i)` and choose the arm with largest sampled **expected net reward** `\tilde{p}_i Rᵢ − Cᵢ`. We **update** `(α_i, β_i)` with observed success/failure.

In [ ]:
class ThompsonSampling(StrategyBase):
    def __init__(self, k, R, C, names, alpha0, beta0):
        super().__init__(k, R, C, names)
        self.alpha0 = alpha0.astype(float)
        self.beta0  = beta0.astype(float)

    def run(self, T, rng):
        k = self.k
        alpha = self.alpha0.copy()
        beta  = self.beta0.copy()

        rewards = np.zeros(T)
        choices = np.zeros(T, dtype=int)
        successes = np.zeros(T)

        for t in range(T):
            p_samp = rng.beta(alpha, beta)  # shape (k,)
            est_mu = p_samp * self.R - self.C
            a = int(np.argmax(est_mu))
            r, s = pull_arm(a, rng)
            rewards[t] = r; choices[t] = a; successes[t] = s
            # update
            if s > 0.5:
                alpha[a] += 1.0
            else:
                beta[a] += 1.0

        post = {"alpha": alpha, "beta": beta}
        return {"rewards": rewards, "choices": choices, "successes": successes, "post": post}

## 5) Configure Strategies

In [ ]:
k = len(p_true)

# Use past data to set initial empirical means for ε‑greedy (via implied expected reward)
# We use posterior means for p_i: (s+a)/(s+f+a+b) with default a=b=1 if no past supplied.
alpha_prior = past_successes + 1.0
beta_prior  = past_failures  + 1.0
p_post_mean = alpha_prior / (alpha_prior + beta_prior)
init_mean_reward = p_post_mean * R - C

eps = 0.1  # exploration rate
eg = EpsilonGreedy(k, R, C, names, epsilon=eps, init_mean=0.0)  # start neutral; it will learn
ucb = UCB1(k, R, C, names)
ts  = ThompsonSampling(k, R, C, names, alpha0=alpha_prior, beta0=beta_prior)
print("Initial posterior mean expected rewards:", (p_post_mean*R - C))

## 6) Single Run per Strategy

In [ ]:
res = {}
for label, strat in [("ε‑greedy", eg), ("UCB1", ucb), ("Thompson", ts)]:
    out = simulate_single(strat, T, seed=rng.integers(0, 1<<31))
    res[label] = out
    cum = np.cumsum(out["rewards"])
    regret = best_mu*np.arange(1, T+1) - cum
    print(f"{label}: final cum. reward = {cum[-1]:.2f} | final regret = {regret[-1]:.2f}")

## 7) Plots — Cumulative Reward and Regret

In [ ]:
# Cumulative reward
fig = plt.figure(figsize=(7,4))
for label, out in res.items():
    plt.plot(np.cumsum(out["rewards"]), label=label)
plt.xlabel("Round"); plt.ylabel("Cumulative reward")
plt.title("Cumulative Reward"); plt.legend(); plt.tight_layout(); plt.show()

# Cumulative regret
fig = plt.figure(figsize=(7,4))
for label, out in res.items():
    regret = best_mu*np.arange(1, T+1) - np.cumsum(out["rewards"])
    plt.plot(regret, label=label)
plt.xlabel("Round"); plt.ylabel("Cumulative regret")
plt.title("Cumulative Regret"); plt.legend(); plt.tight_layout(); plt.show()

## 8) Plots — Allocation Shares (Running Average of Choices)

In [ ]:
def plot_allocation(label, choices, k, w=100):
    fig = plt.figure(figsize=(7,4))
    for a in range(k):
        sel = (choices == a).astype(float)
        rm = running_mean(sel, w=w)
        plt.plot(rm, label=names[a])
    plt.xlabel("Round"); plt.ylabel(f"Share in last ~{w} rounds")
    plt.title(f"Allocation (running share) — {label}")
    plt.legend(); plt.tight_layout(); plt.show()

for label, out in res.items():
    plot_allocation(label, out["choices"], k, w=150)

## 9) Thompson Posterior Summary

In [ ]:
post = res["Thompson"].get("post", None)
if post is not None:
    alpha = post["alpha"]; beta = post["beta"]
    mean_p = alpha / (alpha + beta)
    fig = plt.figure(figsize=(7,4))
    plt.bar(np.arange(k), mean_p)
    plt.xticks(np.arange(k), names)
    plt.ylabel("Posterior mean p_i")
    plt.title("Thompson Posterior Means of Success Rates")
    plt.tight_layout(); plt.show()
    print("Posterior mean E[p]:", mean_p)
else:
    print("No posterior found for Thompson (unexpected).")

## 10) Multi‑Run Experiment (Averaged Curves)

Run repeated simulations to see average behavior and variability.

In [ ]:
N_RUNS = 20
labels = ["ε‑greedy", "UCB1", "Thompson"]
cum_rewards = {lab: np.zeros((N_RUNS, T)) for lab in labels}

for r in range(N_RUNS):
    eg_r = EpsilonGreedy(k, R, C, names, epsilon=eps, init_mean=0.0)
    ucb_r = UCB1(k, R, C, names)
    ts_r  = ThompsonSampling(k, R, C, names, alpha0=alpha_prior, beta0=beta_prior)
    for lab, strat in [("ε‑greedy", eg_r), ("UCB1", ucb_r), ("Thompson", ts_r)]:
        out = simulate_single(strat, T, seed=rng.integers(0, 1<<31))
        cum_rewards[lab][r, :] = np.cumsum(out["rewards"])

fig = plt.figure(figsize=(7,4))
for lab in labels:
    m = np.mean(cum_rewards[lab], axis=0)
    lo = np.percentile(cum_rewards[lab], 10, axis=0)
    hi = np.percentile(cum_rewards[lab], 90, axis=0)
    plt.plot(m, label=lab)
    plt.fill_between(np.arange(T), lo, hi, alpha=0.2)
plt.xlabel("Round"); plt.ylabel("Cumulative reward")
plt.title("Average Cumulative Reward (shaded 10–90%)")
plt.legend(); plt.tight_layout(); plt.show()

## 11) What‑If Knobs

Try changing:
- `p_true`, `R`, `C` to reflect different project portfolios.
- `past_successes`, `past_failures` to encode historical data.
- Horizon `T` and ε in ε‑greedy.
- For UCB1, scaling bounds are automatic via `(R_max, C_max)`.
- For Thompson, priors are `alpha0 = past_successes+1`, `beta0 = past_failures+1`.

## 12) Save Artifacts & Download

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)

# Save the configuration and a single-run trace for reproducibility
single_trace = {
    "names": names,
    "p_true": p_true.tolist(),
    "R": R.tolist(),
    "C": C.tolist(),
    "past_successes": past_successes.tolist(),
    "past_failures": past_failures.tolist(),
    "T": int(T),
}

# Include single-run outputs
for lab, out in res.items():
    single_trace[f"{lab}_cum_reward"] = np.cumsum(out["rewards"]).tolist()
    single_trace[f"{lab}_choices"] = out["choices"].tolist()

import json as _json
with open("artifacts/bandit_single_run.json", "w") as f:
    _json.dump(single_trace, f, indent=2)

# Save multi-run averages
np.savez("artifacts/bandit_multirun_cum_rewards.npz",
         labels=np.array(list(cum_rewards.keys()), dtype=object),
         **{f"CR_{lab}": cum_rewards[lab] for lab in cum_rewards})

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 13) Appendix — Notes & Extensions

- Add **budget per round** and allow **batch funding** of multiple arms with knapsack‑style constraints.
- Model **uncertain payoffs** `Rᵢ` (e.g., log‑normal) and place conjugate priors over *returns*, not only success rates.
- Use **Bayes‑UCB** or **KL‑UCB** for Bernoulli bandits; add **discounting** for non‑stationary environments.
- Switch to **contextual bandits**: include project features or macro signals to condition the policy.
